In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('datasets/courses.csv')
df.head(3)

,Unnamed: 0,Title,URL,Short Intro,Category,Sub-Category,Course Type,Language,Subtitle Languages,Skills,...,Course Short Intro,Weekly study,Premium course,What's include,Rank,Created by,Program,Number of ratings,Price,COURSE CATEGORIES
0,0,Machine Learning Specialization,https://www.coursera.org/specializations/machi...,#BreakIntoAI with Machine Learning Specializat...,Data Science,Machine Learning,Specialization,English,Subtitles: English,"Decision Trees, Artificial Neural Network, Log...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Introduction to Data Science Specialization,https://www.coursera.org/specializations/intro...,Launch your career in data science. Gain found...,Data Science,Data Analysis,Specialization,English,"Subtitles: English, Arabic, French, Portuguese...","Data Science, Relational Database Management S...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Data Science Fundamentals with Python and SQL ...,https://www.coursera.org/specializations/data-...,Build the Foundation for your Data Science car...,Data Science,Data Analysis,Specialization,English,"Subtitles: English, Arabic, French, Portuguese...","Data Science, Github, Python Programming, Jupy...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
columns = ['Title', 'URL', 'Category', 'Sub-Category', 'Language', 'Instructors', 'Level']
df = df[columns]

df = df.fillna("Not available")
df

,Title,URL,Category,Sub-Category,Language,Instructors,Level
0,Machine Learning Specialization,https://www.coursera.org/specializations/machi...,Data Science,Machine Learning,English,"Andrew Ng ,Eddy Shyu ,Aarti Bagul ,Geoff Ladwig ,",Not available
1,Introduction to Data Science Specialization,https://www.coursera.org/specializations/intro...,Data Science,Data Analysis,English,"Rav Ahuja ,Alex Aklson ,Aije Egwaikhide ,Svetl...",Not available
2,Data Science Fundamentals with Python and SQL ...,https://www.coursera.org/specializations/data-...,Data Science,Data Analysis,English,"Aije Egwaikhide ,Svetlana Levitan ,Romeo Kienz...",Not available
3,Key Technologies for Business Specialization,https://www.coursera.org/specializations/key-t...,Business,Business Essentials,English,"Rav Ahuja ,Alex Aklson ,",Not available
4,Deep Learning Specialization,https://www.coursera.org/specializations/deep-...,Data Science,Machine Learning,English,"Andrew Ng ,Kian Katanforoosh ,Younes Bensouda ...",Not available
...,...,...,...,...,...,...,...
8087,Certified Scrum Master,https://www.simplilearn.com/certified-scrum-ma...,Not available,Not available,Not available,Not available,Not available
8088,Salesforce Basics Course for Beginners,https://www.simplilearn.com/salesforce-course?...,Not available,Not available,Not available,Not available,Not available
8089,ICP-ACC (ICAgile Certified Agile Coaching) Cer...,https://www.simplilearn.com/certified-agile-co...,Not available,Not available,Not available,Not available,Not available
8090,/irisprodflip456,https://www.simplilearn.com/irisprodflip456?tag=,Not available,Not available,Not available,Not available,Not available


In [4]:
import re
def clean_text(text):
    if isinstance(text, str):
        text = text.strip()                     # remove leading/trailing spaces
        text = re.sub(r'\s+', ' ', text)        # replace multiple spaces with one
        text = re.sub(r'[^\w\s:/.\-]', '', text) # remove unwanted characters but keep useful ones like ':', '/', '.', '-', etc.
        return text
    return text

for col in df.columns:
    df[col] = df[col].apply(clean_text)

df = df.drop_duplicates()
df

,Title,URL,Category,Sub-Category,Language,Instructors,Level
0,Machine Learning Specialization,https://www.coursera.org/specializations/machi...,Data Science,Machine Learning,English,Andrew Ng Eddy Shyu Aarti Bagul Geoff Ladwig,Not available
1,Introduction to Data Science Specialization,https://www.coursera.org/specializations/intro...,Data Science,Data Analysis,English,Rav Ahuja Alex Aklson Aije Egwaikhide Svetlana...,Not available
2,Data Science Fundamentals with Python and SQL ...,https://www.coursera.org/specializations/data-...,Data Science,Data Analysis,English,Aije Egwaikhide Svetlana Levitan Romeo Kienzle...,Not available
3,Key Technologies for Business Specialization,https://www.coursera.org/specializations/key-t...,Business,Business Essentials,English,Rav Ahuja Alex Aklson,Not available
4,Deep Learning Specialization,https://www.coursera.org/specializations/deep-...,Data Science,Machine Learning,English,Andrew Ng Kian Katanforoosh Younes Bensouda Mo...,Not available
...,...,...,...,...,...,...,...
8087,Certified Scrum Master,https://www.simplilearn.com/certified-scrum-ma...,Not available,Not available,Not available,Not available,Not available
8088,Salesforce Basics Course for Beginners,https://www.simplilearn.com/salesforce-coursetag,Not available,Not available,Not available,Not available,Not available
8089,ICP-ACC ICAgile Certified Agile Coaching Certi...,https://www.simplilearn.com/certified-agile-co...,Not available,Not available,Not available,Not available,Not available
8090,/irisprodflip456,https://www.simplilearn.com/irisprodflip456tag,Not available,Not available,Not available,Not available,Not available


In [ ]:
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema import Document


df['course_text'] = df['Title'] + " | " + df['Category'] + " | " + df['Sub-Category'] + " | " + df['Instructors']

documents = []

for i, row in df.iterrows():
    text = row['course_text']
    metadata = {
        "title": row['Title'],
        "url": row['URL'],
        "category": row['Category'],
        "sub_category": row['Sub-Category'],
        "language": row['Language'],
        "level": row['Level'],
        "instructors": row['Instructors']
    }
    documents.append(Document(page_content=text, metadata=metadata))

embedding_model = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = Chroma(
    embedding_function=embedding_model,
    persist_directory='embeddings',
    collection_name='course_embeddings'
    )
vectorstore.add_documents(documents=documents)

C:\Users\ritam\AppData\Local\Temp\ipykernel_16328\2281088994.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['course_text'] = df['Title'] + " | " + df['Category'] + " | " + df['Sub-Category'] + " | " + df['Instructors']
C:\Users\ritam\AppData\Local\Temp\ipykernel_16328\2281088994.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2

['b07d42f3-0672-4641-8786-fa51e66df769',
 '59a51e21-a30b-4a13-974d-fdae643a9c04',
 '109b72e1-376d-4280-8d73-9d5d7f4054cd',
 '47705a06-90f7-4137-a294-1fbe1d1a5481',
 '8f305039-f27e-4d74-9b10-e6d2288450cf',
 '1ddbc3cc-c27c-4992-a3f9-a8f23a35f587',
 '88fac6dd-1ba3-4806-8dc5-a0ddb76fa787',
 'aa1394c2-63ed-466d-9a9f-dab3d8243761',
 'e909ede8-3268-43b4-b2fb-f718c3ad4f94',
 'b799cd97-8cb4-4b28-831e-bb19bb81841e',
 '0f7f8bb2-97d8-42b3-9466-14ba0d92a72a',
 '31e48db8-7917-423a-ac32-60a053d1a7ed',
 'b461b401-032b-48c0-8e56-b0bf6b26a51d',
 '15ea91ef-fe07-4d8c-b0c9-0215dbf0eff4',
 '94806036-013b-4c50-8e6d-62a885dcc7b6',
 '30c5ebe0-8c34-4404-8fdf-c8ce46f3a4e2',
 'd53b55c0-224b-4b15-b28b-25fb732ca2b9',
 '50023365-286d-41ab-b821-059af0e362cf',
 '33299748-8549-44f2-a48e-8a64ac3de09a',
 '60bceeca-3939-47e0-980e-e0ef31cee72a',
 '58ee1ab8-de09-4657-bd56-0a6b38e227b9',
 '41da6c8a-df83-4911-adc3-bd78b67a5bf2',
 'cbfa49b1-1afc-4473-80cd-852329c6f56f',
 'ee7c166f-37ff-4ce6-9ebb-261e9374a885',
 '897624be-e4ee-

In [3]:
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma

embedding_model = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = Chroma(
    embedding_function=embedding_model,
    persist_directory='embeddings',
    collection_name='course_embeddings'
    )
vectorstore.similarity_search(
    query='Some beginner level courses on Mongodb',
    k=10
)

[Document(metadata={'language': 'Not available', 'sub_category': 'Not available', 'instructors': 'Not available', 'category': 'Not available', 'url': 'https://www.udacity.com/course/data-wrangling-with-mongodb--ud032', 'level': 'Intermediate', 'title': 'Data Wrangling with MongoDB'}, page_content='Data Wrangling with MongoDB | Not available | Not available | Not available'),
 Document(metadata={'level': 'Not available', 'sub_category': 'Data Management', 'instructors': 'Rav Ahuja', 'language': 'English', 'url': 'https://www.coursera.org/learn/introduction-to-nosql-databasesspecializationnosql-big-data-and-spark-foundations', 'category': 'Information Technology', 'title': 'Introduction to NoSQL Databases'}, page_content='Introduction to NoSQL Databases | Information Technology | Data Management | Rav Ahuja'),
 Document(metadata={'title': 'Database Management Essentials', 'level': 'Not available', 'language': 'English', 'instructors': 'Michael Mannino', 'url': 'https://www.coursera.org/l

In [19]:
results = vectorstore.similarity_search_with_score(
    query='Some beginner level courses on SQL',
    k=10
)
for doc in results:
    print(f"Title: {doc[0].metadata['title']}")
    print(f"Category: {doc[0].metadata['category']}")
    print(f"URL: {doc[0].metadata['url']}")
    print("---")

Title: Oracle SQL Basics
Category: Computer Science
URL: https://www.coursera.org/learn/oracle-sql-basicsspecializationoracle-sql-databases
---
Title: Learn SQL Basics for Data Science Specialization
Category: Data Science
URL: https://www.coursera.org/specializations/learn-sql-basics-data-science
---
Title: Oracle SQL Proficiency
Category: Computer Science
URL: https://www.coursera.org/learn/oracle-sql-proficiencyspecializationoracle-sql-databases
---
Title: SQL for Data Science
Category: Data Science
URL: https://www.coursera.org/learn/sql-for-data-science
---
Title: SQL for Data Science
Category: Data Science
URL: https://www.coursera.org/learn/sql-for-data-sciencespecializationlearn-sql-basics-data-science
---
Title: Introduction to Structured Query Language SQL
Category: Computer Science
URL: https://www.coursera.org/learn/intro-sqlspecializationweb-applications
---
Title: Databases and SQL for Data Science with Python
Category: Data Science
URL: https://www.coursera.org/learn/sql

In [ ]:
import torch 

: 